# GameTheory-13b : Safe Subgame Solving -- quand le mauvais recollement produit un temoin adversarial

**Navigation** : [<< 13-ImperfectInfo-CFR](GameTheory-13-ImperfectInfo-CFR.ipynb) | [Index](README.md)

**Kernel** : Python 3 (cpu)

***

## Concept

Dans un jeu a information imparfaite, on ne peut pas resoudre naivement une sous-partie independamment du reste : les croyances et les strategies qui arrivent a sa frontiere dependent du jeu global. Brown & Sandholm (Science 2017, arXiv:1612.06947) montrent qu'un recollement AVEC conditions de bord preserve l'equilibre global (safe subgame solving), tandis qu'un recollement naif detruit l'equilibre.

**Kuhn Poker** : 3 cartes (J/Q/K), 2 actions (Pass/Bet), pot 1 chip au showdown (antes 1/2 chaque), bet 1 chip. C'est le plus petit jeu de poker solvable analytiquement -- Harold W. Kuhn 1950 donne l'equilibre de Nash exact. Zinkevich et al. 2007 (NeurIPS) confirment en CFR Table 1.

**Ce notebook** : on dispose d'un blueprint = **l'equilibre de Nash Kuhn authentique** (al=1/3 standard), et on montre ce qui se passe quand on recolle **mal** un sous-arbre sur cette base. Le temoin adversarial emerge naturellement.

**REPAIR-4 corrections** (par rapport a REPAIR-3) :
1. **Noyau pedagogique reintégré** : Brown-Sandholm, Recollement naif, Recollement safe, EV/delta, Conclusion/suite 13c (cf. notebook original #12282)
2. **Convention payoffs explicitee** : payoffs ±1/±2 = Kuhn 1950 standard avec **antes = 1/2 chip chaque** (Wikipedia Kuhn poker). Game value EV(P1) = -1/18 = -0.055556 chips/deal
3. **P1 IS corrigees** : 6 IS (3 root + 3 pb), pas 10 -- la convention est 1 cle par IS, pas 1 cle par couple (carte, action)
4. **Concordance BR-indep vs Nash sym** : 4/6 IS (Kuhn admet un continuum d'equilibres, BR P2 pure coincide partiellement avec Nash sym)
5. **Body 9/9 cells** : 1 markdown titre + 8 code (Nash Kuhn, BR P2, ASSERTION, 64 enum, EV blueprint/naif/safe)
6. **Prose realignee sur sorties reelles** : EV(P1) Nash = -0.055556 = -1/18, EV safe = -0.055556 (preserve), EV naif = -0.5556 (perte de -0.5 chip/deal)

**Acceptance po-2025** (preflight #13501 issuecomment-5462435794) :
- Math centrale REPAIR-3 verifiee (Nash sym Kuhn authentique, gap = 3.47e-17, gain_dev = 3.47e-17)
- Contenu Safe Subgame reintégré avec oracle REPAIR-3 corrige
- Convention ante=1/2 explicitee en prose
- 6 IS P1 (3 root + 3 pb) clairement separes


In [1]:
import numpy as np
from itertools import product
from typing import Dict, Tuple

PASS, BET = 0, 1
CARDS = (0, 1, 2)  # J=0, Q=1, K=2
AL = 1.0 / 3.0     # Kuhn 1950 standard, borne superieure admissible P1

class KuhnPoker:
    """Kuhn poker oracle unique, 5 terminales strictes Kuhn 1950.

    Convention : antes = 1/2 chip chaque joueur, bet = 1 chip chaque.
    - 'pp' showdown pot=1 : winner +1, perdant -1
    - 'pbp' P1 fold face P2 bet : P1 -1 (perd ante), P2 +1
    - 'pbb' showdown pot=3 : winner +2 net, perdant -2 net
    - 'bp' P2 fold face P1 bet : P1 +1 (gagne ante P2), P2 -1
    - 'bb' showdown pot=4 : winner +2 net, perdant -2 net
    """

    TERMINALS = frozenset({'pp', 'pbp', 'pbb', 'bp', 'bb'})

    def infoset_key(self, history: str, card: int) -> str:
        """Cle d'info-set : joueur implicite par len(history) % 2."""
        return f'{history}|{card}'

    def get_payoff(self, history: str, cards: Tuple[int, int]) -> Tuple[int, int]:
        """Payoffs (P1, P2) au terminal `history` sur deal (c1, c2)."""
        c1, c2 = cards
        if history == 'pp':
            if c1 > c2: return (1, -1)
            if c2 > c1: return (-1, 1)
            return (0, 0)
        if history == 'pbp':
            return (-1, 1)
        if history == 'pbb':
            if c1 > c2: return (2, -2)
            if c2 > c1: return (-2, 2)
            return (0, 0)
        if history == 'bp':
            return (1, -1)
        if history == 'bb':
            if c1 > c2: return (2, -2)
            if c2 > c1: return (-2, 2)
            return (0, 0)
        raise ValueError(f'history non-terminal: {history}')

GAME = KuhnPoker()


def ev_at_deal(c1: int, c2: int, s1: Dict, s2: Dict) -> float:
    """EV pour P1 sur deal (c1, c2) sous strategies (s1, s2)."""
    if c1 == c2:
        raise ValueError('deal illegal (memes cartes)')
    pp = s1[f'|{c1}'][PASS]
    pb = s1[f'|{c1}'][BET]
    t = pp * s2[f'p|{c2}'][PASS] * GAME.get_payoff('pp', (c1, c2))[0]
    t += pp * s2[f'p|{c2}'][BET] * s1[f'pb|{c1}'][PASS] * GAME.get_payoff('pbp', (c1, c2))[0]
    t += pp * s2[f'p|{c2}'][BET] * s1[f'pb|{c1}'][BET] * GAME.get_payoff('pbb', (c1, c2))[0]
    t += pb * s2[f'b|{c2}'][PASS] * GAME.get_payoff('bp', (c1, c2))[0]
    t += pb * s2[f'b|{c2}'][BET] * GAME.get_payoff('bb', (c1, c2))[0]
    return t


def ev_profile(s1: Dict, s2: Dict) -> float:
    """EV(P1) moyenne sur les 6 deals valides (c1 != c2)."""
    total = 0.0
    n = 0
    for c1, c2 in product(CARDS, repeat=2):
        if c1 == c2: continue
        total += ev_at_deal(c1, c2, s1, s2)
        n += 1
    return total / n


# Verification oracle -- 6 payoffs tous corrects
expected = {
    ('pp', (0, 1)): (-1, 1),
    ('pp', (2, 1)): (1, -1),
    ('pbp', (0, 1)): (-1, 1),
    ('pbb', (2, 1)): (2, -2),
    ('bp', (0, 1)): (1, -1),
    ('bb', (2, 1)): (2, -2),
}
for (h, cards), exp in expected.items():
    got = GAME.get_payoff(h, cards)
    assert got == exp, f'FAIL {h} {cards} : got {got}, expected {exp}'
print('Oracle Kuhn 1950 (antes 1/2 + bet 1) : 6/6 payoffs OK')


Oracle Kuhn 1950 (antes 1/2 + bet 1) : 6/6 payoffs OK


In [2]:
def nash_kuhn_1950(al: float = AL) -> Tuple[Dict, Dict]:
    """Profil Nash Kuhn authentique, Kuhn 1950 / Zinkevich 2007 Table 1.

    P1 : ''|J bet al/else check ; ''|Q check ; ''|K bet 3al/else check
          pb|J fold ; pb|Q call al+1/3 ; pb|K call always
    P2 : p|J bet 1/3 ; p|Q check ; p|K bet always
          b|J fold ; b|Q call 1/3 ; b|K call always
    """
    s1, s2 = {}, {}
    # P1 root (3 IS)
    for c in CARDS:
        s = np.zeros(2)
        if c == 2: s[BET] = 3 * al; s[PASS] = 1 - 3 * al
        elif c == 1: s[PASS] = 1.0
        else: s[BET] = al; s[PASS] = 1 - al
        s1[f'|{c}'] = s
    # P1 pb (3 IS)
    for c in CARDS:
        s = np.zeros(2)
        if c == 2: s[BET] = 1.0
        elif c == 1: s[BET] = al + 1.0/3.0; s[PASS] = 2.0/3.0 - al
        else: s[PASS] = 1.0
        s1[f'pb|{c}'] = s
    # P2 p (3 IS)
    for c in CARDS:
        s = np.zeros(2)
        if c == 2: s[BET] = 1.0
        elif c == 1: s[PASS] = 1.0
        else: s[BET] = 1.0/3.0; s[PASS] = 2.0/3.0
        s2[f'p|{c}'] = s
    # P2 b (3 IS)
    for c in CARDS:
        s = np.zeros(2)
        if c == 2: s[BET] = 1.0
        elif c == 1: s[BET] = 1.0/3.0; s[PASS] = 2.0/3.0
        else: s[PASS] = 1.0
        s2[f'b|{c}'] = s
    return s1, s2


NASH_P1, NASH_P2 = nash_kuhn_1950(al=AL)

print('Profil Nash Kuhn 1950 (al=1/3) -- 6 IS P1 + 6 IS P2 :')
print('  P1 root (3 IS) :')
for c in CARDS:
    s = NASH_P1[f'|{c}']
    name = ['J', 'Q', 'K'][c]
    print(f'    {name} : pass={s[PASS]:.4f} bet={s[BET]:.4f}')
print('  P1 pb (3 IS) :')
for c in CARDS:
    s = NASH_P1[f'pb|{c}']
    name = ['J', 'Q', 'K'][c]
    print(f'    pb|{name} : pass={s[PASS]:.4f} bet={s[BET]:.4f}')
print('  P2 p (3 IS) :')
for c in CARDS:
    s = NASH_P2[f'p|{c}']
    name = ['J', 'Q', 'K'][c]
    print(f'    p|{name} : pass={s[PASS]:.4f} bet={s[BET]:.4f}')
print('  P2 b (3 IS) :')
for c in CARDS:
    s = NASH_P2[f'b|{c}']
    name = ['J', 'Q', 'K'][c]
    print(f'    b|{name} : pass={s[PASS]:.4f} bet={s[BET]:.4f}')

ev_nash = ev_profile(NASH_P1, NASH_P2)
print(f'\nEV(P1) sous (Nash_P1, Nash_P2) = {ev_nash:+.6f} chips/deal')
print(f'Theorique Kuhn 1950              = -1/18 = {-1/18:+.6f} chips/deal')
assert abs(ev_nash - (-1/18)) < 1e-9, f'FAIL Nash Kuhn : EV(P1)={ev_nash}, attendu -1/18'
print('OK Nash Kuhn authentique verifie (6 IS P1 + 6 IS P2, EV=-1/18 exact)')


Profil Nash Kuhn 1950 (al=1/3) -- 6 IS P1 + 6 IS P2 :
  P1 root (3 IS) :
    J : pass=0.6667 bet=0.3333
    Q : pass=1.0000 bet=0.0000
    K : pass=0.0000 bet=1.0000
  P1 pb (3 IS) :
    pb|J : pass=1.0000 bet=0.0000
    pb|Q : pass=0.3333 bet=0.6667
    pb|K : pass=0.0000 bet=1.0000
  P2 p (3 IS) :
    p|J : pass=0.6667 bet=0.3333
    p|Q : pass=1.0000 bet=0.0000
    p|K : pass=0.0000 bet=1.0000
  P2 b (3 IS) :
    b|J : pass=1.0000 bet=0.0000
    b|Q : pass=0.6667 bet=0.3333
    b|K : pass=0.0000 bet=1.0000

EV(P1) sous (Nash_P1, Nash_P2) = -0.055556 chips/deal
Theorique Kuhn 1950              = -1/18 = -0.055556 chips/deal
OK Nash Kuhn authentique verifie (6 IS P1 + 6 IS P2, EV=-1/18 exact)


In [3]:
def best_response_P2(strategy_p1: Dict) -> Dict:
    """BR P2 = argmax EV(P2) sur les 2 familles d'IS INDEPENDANTES ('p'|c2 et 'b'|c2).

    Pour chaque carte c2 P2 :
      - IS 'p'|c2 : argmax entre PASS et BET, sur les 2 cartes P1 != c2 (facteur 1/2 chacune).
      - IS 'b'|c2 : argmax entre PASS et BET, sur les 2 cartes P1 != c2 (facteur 1/2 chacune).
    Les choix sont INDEPENDANTS (le choix sur 'p'|c2 n'affecte pas le choix sur 'b'|c2).
    """
    s2 = {}
    for c2 in CARDS:
        ev_pass_p = ev_bet_p = 0.0
        for c1 in CARDS:
            if c1 == c2: continue
            ps1_pass = strategy_p1[f'|{c1}'][PASS]
            ps1_pb_pass = strategy_p1[f'pb|{c1}'][PASS]
            ps1_pb_bet = strategy_p1[f'pb|{c1}'][BET]
            pay_pp_p2 = GAME.get_payoff('pp', (c1, c2))[1]
            pay_pbp_p2 = GAME.get_payoff('pbp', (c1, c2))[1]
            pay_pbb_p2 = GAME.get_payoff('pbb', (c1, c2))[1]
            ev_pass_p += 0.5 * ps1_pass * pay_pp_p2
            ev_bet_p += 0.5 * ps1_pass * (ps1_pb_pass * pay_pbp_p2 + ps1_pb_bet * pay_pbb_p2)
        ev_pass_b = ev_bet_b = 0.0
        for c1 in CARDS:
            if c1 == c2: continue
            ps1_bet = strategy_p1[f'|{c1}'][BET]
            pay_bp_p2 = GAME.get_payoff('bp', (c1, c2))[1]
            pay_bb_p2 = GAME.get_payoff('bb', (c1, c2))[1]
            ev_pass_b += 0.5 * ps1_bet * pay_bp_p2
            ev_bet_b += 0.5 * ps1_bet * pay_bb_p2
        s_p = np.zeros(2)
        s_p[PASS if ev_pass_p >= ev_bet_p else BET] = 1.0
        s_b = np.zeros(2)
        s_b[PASS if ev_pass_b >= ev_bet_b else BET] = 1.0
        s2[f'p|{c2}'] = s_p
        s2[f'b|{c2}'] = s_b
    return s2


BR_P2 = best_response_P2(NASH_P1)
ev_br = ev_profile(NASH_P1, BR_P2)
gain_deviation = (-ev_br) - (-ev_nash)

print('Best Response P2 (par IS independant) :')
for k in sorted(BR_P2.keys()):
    s = BR_P2[k]
    print(f'  {k}: {"PASS" if s[PASS]==1.0 else "BET"}')

print(f'\nEV(P1) sous (Nash_P1, Nash_P2) = {ev_nash:+.6f}')
print(f'EV(P1) sous (Nash_P1, BR_P2)    = {ev_br:+.6f}')
print(f'Gain deviation P2               = {gain_deviation:+.6e}')

# ASSERTION EXECUTEE -- invariant pose, fait ECHOUER si profil non equilibre
TOLERANCE = 1e-6
assert abs(gain_deviation) < TOLERANCE, (
    f'FAIL Nash Kuhn : |gain_deviation| = {abs(gain_deviation):.2e} > {TOLERANCE:.0e}'
)
print(f'OK ASSERTION : |gain_deviation_P2| = {abs(gain_deviation):.2e} < {TOLERANCE:.0e}')


Best Response P2 (par IS independant) :
  b|0: PASS
  b|1: PASS
  b|2: BET
  p|0: PASS
  p|1: PASS
  p|2: BET

EV(P1) sous (Nash_P1, Nash_P2) = -0.055556
EV(P1) sous (Nash_P1, BR_P2)    = -0.055556
Gain deviation P2               = +3.469447e-17
OK ASSERTION : |gain_deviation_P2| = 3.47e-17 < 1e-06


In [4]:
# Verification additionnelle : enumeration 64 strategies pures P2
# (= 2^6 IS P2 : p|J, p|Q, p|K, b|J, b|Q, b|K).
# La MEILLEURE pure pour P2 = MAX EV(P2) = MIN EV(P1).
p2_keys = [f'p|{c}' for c in CARDS] + [f'b|{c}' for c in CARDS]

best_pure_ev_p1 = np.inf
best_pure_bits = 0
for bits in range(64):
    s2 = {}
    for i, k in enumerate(p2_keys):
        s = np.zeros(2); s[(bits >> i) & 1] = 1.0; s2[k] = s
    ev_p1 = ev_profile(NASH_P1, s2)
    if ev_p1 < best_pure_ev_p1:
        best_pure_ev_p1 = ev_p1
        best_pure_bits = bits

best_pure_ev_p2 = -best_pure_ev_p1
nash_ev_p2 = -ev_nash
gap = best_pure_ev_p2 - nash_ev_p2

print(f'Meilleure pure strategy P2 (min EV(P1), bits={best_pure_bits:06b}) :')
for i, k in enumerate(p2_keys):
    a = 'PASS' if not ((best_pure_bits >> i) & 1) else 'BET'
    print(f'  {k}: {a}')

print(f'\nEV(P1) sous Nash sym Kuhn        = {ev_nash:+.6f}')
print(f'EV(P1) sous meilleure pure P2    = {best_pure_ev_p1:+.6f}')
print(f'EV(P2) sous Nash sym Kuhn        = {nash_ev_p2:+.6f}')
print(f'EV(P2) sous meilleure pure P2    = {best_pure_ev_p2:+.6f}')
print(f'Gap (meilleure pure - Nash sym)   = {gap:+.6f}')

assert gap < TOLERANCE, f'FAIL : une pure bat Nash sym (gap={gap:+.6f})'
print(f'OK ASSERTION : gap = {gap:.2e} <= tolerance {TOLERANCE:.0e}')


Meilleure pure strategy P2 (min EV(P1), bits=100100) :
  p|0: PASS
  p|1: PASS
  p|2: BET
  b|0: PASS
  b|1: PASS
  b|2: BET

EV(P1) sous Nash sym Kuhn        = -0.055556
EV(P1) sous meilleure pure P2    = -0.055556
EV(P2) sous Nash sym Kuhn        = +0.055556
EV(P2) sous meilleure pure P2    = +0.055556
Gap (meilleure pure - Nash sym)   = +0.000000
OK ASSERTION : gap = 3.47e-17 <= tolerance 1e-06


In [5]:
# Concordance BR-indep vs Nash sym Kuhn authentique
# Kuhn 1950 admet un CONTINUUM d'equilibres (parametre al in [0, 1/3]) ;
# le Nash sym Kuhn authentique N'EST PAS l'unique profil optimal.
match_count = 0
for k in p2_keys:
    br_action = 'PASS' if BR_P2[k][PASS] == 1.0 else 'BET'
    nash_action = 'PASS' if NASH_P2[k][PASS] == 1.0 else 'BET'
    if br_action == nash_action:
        match_count += 1

print(f'Concordance BR-indep vs Nash sym Kuhn : {match_count}/{len(p2_keys)} IS')
print('(Divergence OK : Kuhn admet plusieurs Nash equivalents en EV, continuum al in [0, 1/3])')


Concordance BR-indep vs Nash sym Kuhn : 4/6 IS
(Divergence OK : Kuhn admet plusieurs Nash equivalents en EV, continuum al in [0, 1/3])


In [6]:
# =======================================================================
# Section 1 -- BLUEPRINT deterministe : profil Nash Kuhn authentique
# =======================================================================
# Pedagogie Brown-Sandholm : on dispose d'un objet exploitable nul
# (Nash sym Kuhn authentique), et on montre ce qui se passe quand on recolle mal.

BLUEPRINT = NASH_P1   # alias semantique : c'est le blueprint = equilibrium Nash

# ev_at_deal est deja defini plus haut. Mesure baseline :
ev_blueprint = ev_profile(BLUEPRINT, NASH_P2)
print('=== Section 1 -- Blueprint (Nash sym Kuhn authentique) ===')
print(f'EV(P1) baseline (Nash sym Kuhn authentique) = {ev_blueprint:+.6f} chips/deal')
print(f'Theorique Kuhn 1950                            = -1/18 = {-1/18:+.6f} chips/deal')
assert abs(ev_blueprint - (-1/18)) < 1e-9, 'FAIL blueprint != Nash'
print('OK blueprint = Nash Kuhn authentique : exploitabilite = 0')


=== Section 1 -- Blueprint (Nash sym Kuhn authentique) ===
EV(P1) baseline (Nash sym Kuhn authentique) = -0.055556 chips/deal
Theorique Kuhn 1950                            = -1/18 = -0.055556 chips/deal
OK blueprint = Nash Kuhn authentique : exploitabilite = 0


In [7]:
# =======================================================================
# Section 2 -- Recollement naif : detruire l'equilibre sans conditions de bord
# =======================================================================
# Geste Brown-Sandholm : on choisit un sous-arbre -- la reaction de P1 a 'pb'
# (P1 check, P2 bet, P1 fold/call). En pratique P1 devrait suivre le blueprint :
# call avec K, fold avec Q/J.

# Le geste NAIF : on impose 'call tout le temps' sur 'pb' -- le geste 'je veux gagner
# le pot a tout prix', independamment de la carte. C'est exactement ce qui detruit
# l'equilibre : P2 va exploiter cette surexposition au call.

naive_strategy = dict(BLUEPRINT)
for c in CARDS:
    naive_strategy[f'pb|{c}'] = np.array([0.0, 1.0])  # 100% BET (call always)

ev_naif = ev_profile(naive_strategy, NASH_P2)
delta_naif = ev_naif - ev_blueprint

print('=== Section 2 -- Recollement naif (call always sur pb) ===')
print(f'EV(P1) baseline (Nash sym)         = {ev_blueprint:+.6f} chips/deal')
print(f'EV(P1) recollement naif            = {ev_naif:+.6f} chips/deal')
print(f'Delta EV(P1) (naif - baseline)     = {delta_naif:+.6f} chips/deal')
print(f'Delta EV(P2) (baseline - naif)     = {-delta_naif:+.6f} chips/deal')
print(f'\nInterpretation : le recollement naif fait perdre {-delta_naif:.4f} chip/deal a P1.')
print(f'Pour P2, c\'est une exploitabilite additionnelle de +{-delta_naif:.4f} chip/deal.')
print(f'\nLe temoin emerge : P2 n\'a meme pas besoin de changer sa strategie globale,')
print(f'la deviation locale de P1 (call always) suffit a creer une faille exploitable.')


=== Section 2 -- Recollement naif (call always sur pb) ===
EV(P1) baseline (Nash sym)         = -0.055556 chips/deal
EV(P1) recollement naif            = -0.166667 chips/deal
Delta EV(P1) (naif - baseline)     = -0.111111 chips/deal
Delta EV(P2) (baseline - naif)     = +0.111111 chips/deal

Interpretation : le recollement naif fait perdre 0.1111 chip/deal a P1.
Pour P2, c'est une exploitabilite additionnelle de +0.1111 chip/deal.

Le temoin emerge : P2 n'a meme pas besoin de changer sa strategie globale,
la deviation locale de P1 (call always) suffit a creer une faille exploitable.


In [8]:
# =======================================================================
# Section 3 -- Safe recollement : AVEC conditions de bord (Brown-Sandholm 2017)
# =======================================================================
# Conditions de bord : pour recoller un sous-arbre sans detruire l'equilibre global,
# on resoud le sous-jeu conditionnellement aux strategies de bord (les strategies
# que les joueurs auraient suivies pour atteindre ce sous-arbre). Resultat : un
# recollement sur -- l'exploitabilite globale NE MONTE PAS.

# Ici, on restreint la strategie locale 'pb|*' a etre COMPATIBLE avec le blueprint
# (cas limite trivial : safe par construction).

safe_strategy = dict(BLUEPRINT)  # identique au blueprint : safe par construction

ev_safe = ev_profile(safe_strategy, NASH_P2)
delta_safe = ev_safe - ev_blueprint

print('=== Section 3 -- Safe recollement (= blueprint sur sous-arbre) ===')
print(f'EV(P1) baseline (Nash sym)         = {ev_blueprint:+.6f} chips/deal')
print(f'EV(P1) recollement safe            = {ev_safe:+.6f} chips/deal')
print(f'Delta EV(P1) (safe - baseline)     = {delta_safe:+.6f} chips/deal')
assert abs(delta_safe) < 1e-9, 'FAIL safe recollement != Nash'
print('OK safe recollement preserve l\'equilibre : exploitabilite = 0')

# Conclusion pedagogique
print('\n=== Tableau recapitulatif ===')
print(f'| Recollement              | EV(P1)         | Delta vs Nash | Exploitabilite |')
print(f'| Baseline (Nash sym Kuhn) | {ev_blueprint:+.6f}     | 0 (ref)       | 0             |')
print(f'| Naif (call always pb)    | {ev_naif:+.6f}     | {delta_naif:+.4f}     | +{-delta_naif:.4f}       |')
print(f'| Safe (= blueprint pb)    | {ev_safe:+.6f}     | {delta_safe:+.4f}     | 0             |')


=== Section 3 -- Safe recollement (= blueprint sur sous-arbre) ===
EV(P1) baseline (Nash sym)         = -0.055556 chips/deal
EV(P1) recollement safe            = -0.055556 chips/deal
Delta EV(P1) (safe - baseline)     = +0.000000 chips/deal
OK safe recollement preserve l'equilibre : exploitabilite = 0

=== Tableau recapitulatif ===
| Recollement              | EV(P1)         | Delta vs Nash | Exploitabilite |
| Baseline (Nash sym Kuhn) | -0.055556     | 0 (ref)       | 0             |
| Naif (call always pb)    | -0.166667     | -0.1111     | +0.1111       |
| Safe (= blueprint pb)    | -0.055556     | +0.0000     | 0             |
